# Auto-Generate Column Comments with LLM

This notebook scans all tables in a Unity Catalog schema, identifies columns **without** comments, generates descriptive comments using an LLM via `ai_query`, and applies them via `ALTER TABLE` or `ALTER VIEW`.

**Existing comments are never overwritten.** Streaming tables and materialized views are detected and skipped — their comments should be managed in the pipeline definition.

In [0]:
%python
dbutils.widgets.text("catalog_name", "", "Catalog Name")
dbutils.widgets.text("schema_name", "", "Schema Name")

catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")

print(f"Target: {catalog_name}.{schema_name}")

In [0]:
%python
# Get all columns from all object types (tables and views) that lack comments
# Then separate pipeline-managed tables from regular tables
all_columns = spark.sql(f"""
    SELECT c.table_name, c.column_name, c.data_type, c.comment, t.table_type
    FROM {catalog_name}.information_schema.columns c
    JOIN {catalog_name}.information_schema.tables t
      ON c.table_catalog = t.table_catalog
      AND c.table_schema = t.table_schema
      AND c.table_name = t.table_name
    WHERE c.table_schema = '{schema_name}'
      AND (c.comment IS NULL OR TRIM(c.comment) = '')
    ORDER BY c.table_name, c.ordinal_position
""")

# Separate pipeline-managed objects (streaming tables & materialized views) from regular tables
pipeline_columns = all_columns.filter("table_type IN ('STREAMING_TABLE', 'MATERIALIZED_VIEW')")
columns_needing_comments = all_columns.filter("table_type NOT IN ('STREAMING_TABLE', 'MATERIALIZED_VIEW')")

pipeline_table_count = pipeline_columns.select("table_name").distinct().count()
total = columns_needing_comments.count()

if pipeline_table_count > 0:
    pipeline_tables = pipeline_columns.select("table_name", "table_type").distinct().collect()
    print("⚠ The following tables are managed by a pipeline — add comments in the pipeline definition instead:")
    for pt in pipeline_tables:
        print(f"  • {pt['table_name']} ({pt['table_type']})")
    print()

print(f"Found {total} column(s) in regular tables that need comments")
display(columns_needing_comments)

In [0]:
%python
from pyspark.sql.functions import concat, lit, col, expr

if total == 0:
    print("All columns already have comments — nothing to generate.")
else:
    # Build a prompt per column and call the LLM in batch via ai_query
    comments_df = (
        columns_needing_comments
        .select("table_name", "column_name", "data_type", "table_type")
        .withColumn(
            "prompt",
            concat(
                lit("Generate a single concise comment (max 15 words) for a database column named '"),
                col("column_name"),
                lit("' of type '"),
                col("data_type"),
                lit("' in table '"),
                col("table_name"),
                lit("'. Return ONLY the comment text, no quotes or extra formatting.")
            )
        )
        .withColumn(
            "generated_comment",
            expr("ai_query('databricks-meta-llama-3-3-70b-instruct', prompt)")
        )
    )
    display(comments_df.select("table_name", "column_name", "data_type", "table_type", "generated_comment"))

In [0]:
%python
if total == 0:
    print("Nothing to apply.")
else:
    # Collect all generated comments
    rows = comments_df.select("table_name", "column_name", "table_type", "generated_comment").collect()
    success_count = 0
    error_count = 0

    # --- Apply comments to regular tables via ALTER TABLE ---
    table_rows = [r for r in rows if r["table_type"] != "VIEW"]
    for row in table_rows:
        table = row["table_name"]
        column = row["column_name"]
        comment = (row["generated_comment"] or "").replace("'", "''").replace("\n", " ").strip()
        if not comment:
            print(f"⚠ Skipping {table}.{column} — LLM returned empty comment")
            continue
        alter_sql = f"ALTER TABLE `{catalog_name}`.`{schema_name}`.`{table}` ALTER COLUMN `{column}` COMMENT '{comment}'"
        try:
            spark.sql(alter_sql)
            success_count += 1
            print(f"✓ {table}.{column}")
        except Exception as e:
            error_count += 1
            print(f"✗ {table}.{column}: {e}")

    # --- Apply comments to views via CREATE OR REPLACE VIEW ---
    view_rows = [r for r in rows if r["table_type"] == "VIEW"]
    if view_rows:
        from collections import defaultdict
        new_comments = defaultdict(dict)
        for row in view_rows:
            comment = (row["generated_comment"] or "").replace("'", "''").replace("\n", " ").strip()
            if comment:
                new_comments[row["table_name"]][row["column_name"]] = comment

        for view_name, col_comments in new_comments.items():
            try:
                # Get the original view query from information_schema
                view_def = spark.sql(f"""
                    SELECT view_definition
                    FROM {catalog_name}.information_schema.views
                    WHERE table_schema = '{schema_name}' AND table_name = '{view_name}'
                """).collect()[0]["view_definition"]

                # Get ALL columns to preserve existing comments
                all_view_cols = spark.sql(f"""
                    SELECT column_name, comment
                    FROM {catalog_name}.information_schema.columns
                    WHERE table_schema = '{schema_name}' AND table_name = '{view_name}'
                    ORDER BY ordinal_position
                """).collect()

                # Build column list: new LLM comments + preserved existing comments
                col_defs = []
                for vc in all_view_cols:
                    cname = vc["column_name"]
                    existing = vc["comment"]
                    if cname in col_comments:
                        col_defs.append(f"  `{cname}` COMMENT '{col_comments[cname]}'")
                    elif existing:
                        safe = existing.replace("'", "''")
                        col_defs.append(f"  `{cname}` COMMENT '{safe}'")
                    else:
                        col_defs.append(f"  `{cname}`")

                col_list = ",\n".join(col_defs)
                create_sql = f"CREATE OR REPLACE VIEW `{catalog_name}`.`{schema_name}`.`{view_name}` (\n{col_list}\n)\nAS {view_def}"
                spark.sql(create_sql)
                success_count += len(col_comments)
                for cname in col_comments:
                    print(f"✓ {view_name}.{cname} (view)")
            except Exception as e:
                error_count += len(col_comments)
                for cname in col_comments:
                    print(f"✗ {view_name}.{cname}: {e}")

    print(f"\nDone! {success_count} comments added, {error_count} errors.")